# Scraper Kompas.com (Search)

Notebook ini melakukan scraping hasil pencarian Kompas (dengan pagination) dan mengambil **judul, tanggal, dan isi artikel**.

Fitur utama:
- Selector / format pembacaan elemen mengikuti versi Anda (mis. `a.article-link`, `h2`, `div.articlePost-date`, pagination `a.paging__link--last`).
- Checkpoint penyimpanan tiap 100 data (append ke CSV), sehingga jika terjadi error/proses berhenti, data yang sudah terkumpul tetap aman.
- Menampilkan progres seperti: `tersimpan 1469 dari 12999` (jika total hasil bisa dideteksi).


In [5]:
import os
import re
import csv
import time
from urllib.parse import urlparse, parse_qs

import requests
from bs4 import BeautifulSoup

# =========================
# KONFIGURASI
# =========================
QUERY = "politik+indonesia"
SORT = "latest"
SITE_ID = 1
LAST_DATE = "all"

CSV_FILENAME = "kompas_politik_indonesia_articles.csv"
CSV_HEADERS = ["title", "tanggal", "content"]

BATCH_SIZE = 100        # simpan setiap 100 artikel
START_PAGE = 1          # boleh diubah untuk melanjutkan dari page tertentu
SLEEP_SEC = 0.2         # jeda ringan agar lebih stabil

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

# Requests session (lebih efisien)
SESSION = requests.Session()
SESSION.headers.update(HEADERS)


In [2]:
def fetch_soup(url: str, timeout: int = 15, max_retry: int = 5, wait_seconds: int = 120) -> BeautifulSoup:
    """GET url -> BeautifulSoup (lxml) dengan retry.
    Jika terjadi error jaringan (timeout/DNS/Max retries exceeded), tunggu `wait_seconds` lalu coba lagi.
    Raise jika status bukan 200 (setelah raise_for_status), dan skip setelah max_retry gagal (return None).
    """
    attempt = 0
    while attempt < max_retry:
        try:
            r = SESSION.get(url, timeout=timeout)
            r.raise_for_status()
            return BeautifulSoup(r.text, 'lxml')

        except requests.exceptions.RequestException as e:
            attempt += 1
            print(
                f"[ERROR] Request gagal | Percobaan {attempt}/{max_retry}\n"
                f"URL: {url}\n"
                f"{e}\n"
                f"Menunggu {wait_seconds} detik sebelum retry...\n"
            )
            time.sleep(wait_seconds)

    print(f"[SKIP] URL dilewati setelah {max_retry} gagal: {url}")
    return None


def append_rows_to_csv(filename: str, headers: list, rows: list) -> None:
    '''
    Append baris ke CSV.
    - Header ditulis hanya jika file belum ada / kosong.
    - Flush + fsync supaya data benar-benar tersimpan (best effort).
    '''
    if not rows:
        return

    file_exists = os.path.exists(filename) and os.path.getsize(filename) > 0

    with open(filename, 'a', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        if not file_exists:
            w.writerow(headers)
        w.writerows(rows)

        f.flush()
        try:
            os.fsync(f.fileno())
        except Exception:
            pass


def extract_page_from_href(href: str):
    'Ambil angka page dari href (?page=123 atau &page=123).'
    if not href:
        return None

    m = re.search(r'[?&]page=(\d+)', href)
    if m:
        return int(m.group(1))

    qs = parse_qs(urlparse(href).query)
    if 'page' in qs and qs['page']:
        try:
            return int(qs['page'][0])
        except ValueError:
            return None

    return None


In [3]:
def build_search_url(page: int) -> str:
    return (
        "https://search.kompas.com/search?"
        f"q={QUERY}&sort={SORT}&site_id={SITE_ID}&last_date={LAST_DATE}&page={page}"
    )


def detect_last_page(soup: BeautifulSoup):
    '''
    Pagination: sesuai versi Anda -> cari anchor dengan selector:
    - a.paging__link--last
    '''
    last_a = soup.select_one('a.paging__link--last')
    if not last_a:
        return None

    # Prioritas: data-ci-pagination-page
    data_page = last_a.get('data-ci-pagination-page')
    if data_page and str(data_page).isdigit():
        return int(data_page)

    return extract_page_from_href(last_a.get('href'))


def detect_total_results(soup: BeautifulSoup):
    '''
    Best effort untuk mendapatkan total hasil (misal 12999).
    Jika tidak ketemu, return None.

    Struktur HTML Kompas search bisa berubah; fungsi ini defensif,
    dan tidak memengaruhi selector scraping utama.
    '''
    text = soup.get_text(' ', strip=True)

    # contoh pola yang sering muncul: "Ditemukan 12.999" atau "12.999 hasil"
    candidates = [
        r'Ditemukan\s+([\d\.]+)',
        r'([\d\.]+)\s+hasil',
        r'Total\s+([\d\.]+)',
    ]
    for pat in candidates:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            raw = m.group(1)
            try:
                return int(raw.replace('.', ''))
            except ValueError:
                pass
    return None


def get_article_kompas(link: str) -> str:
    '''
    Ambil isi artikel.
    Selector sesuai versi Anda:
    - div.read__content
    - paragraphs: find_all('p')
    '''
    try:
        soup = fetch_soup(link)
        if soup is None:
            return ''
        div_content = soup.find('div', class_='read__content')
        if not div_content:
            return ''

        paragraphs = div_content.find_all('p')
        content = ' '.join([p.get_text(strip=True) for p in paragraphs])
        return content

    except Exception as e:
        print(f"Error artikel di: {link} | {e}")
        return ''


In [4]:
# =========================
# INISIALISASI: ambil last page & total hasil
# =========================
first_url = build_search_url(page=1)
soup_first = fetch_soup(first_url)
if soup_first is None:
    raise RuntimeError('Gagal mengambil halaman pertama setelah retry. Cek koneksi/DNS atau coba lagi nanti.')

last_page_kompas = detect_last_page(soup_first)
total_results = detect_total_results(soup_first)

print("Halaman terakhir terdeteksi:", last_page_kompas)
print("Total hasil terdeteksi:", total_results)


[ERROR] Request gagal | Percobaan 1/5
URL: https://search.kompas.com/search?q=politik+indonesia&sort=latest&site_id=1&last_date=all&page=1
HTTPSConnectionPool(host='search.kompas.com', port=443): Max retries exceeded with url: /search?q=politik+indonesia&sort=latest&site_id=1&last_date=all&page=1 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000021B913EEBA0>: Failed to resolve 'search.kompas.com' ([Errno 11001] getaddrinfo failed)"))
Menunggu 120 detik sebelum retry...

Halaman terakhir terdeteksi: 499
Total hasil terdeteksi: 29139


In [5]:
# =========================
# SCRAPING UTAMA (checkpoint tiap 100 data)
# =========================
if not last_page_kompas:
    raise RuntimeError("Tidak bisa mendeteksi last page (a.paging__link--last).")

saved_count = 0
batch = []

# Jika file sudah ada, hitung jumlah baris yang sudah tersimpan (untuk resume)
if os.path.exists(CSV_FILENAME) and os.path.getsize(CSV_FILENAME) > 0:
    try:
        with open(CSV_FILENAME, 'r', encoding='utf-8') as f:
            # -1 untuk header
            saved_count = max(sum(1 for _ in f) - 1, 0)
    except Exception:
        saved_count = 0

counter = 0

for page in range(START_PAGE, last_page_kompas + 1):
    url = build_search_url(page=page)
    print(f"\nScraping halaman ke-{page}: {url}")

    try:
        soup = fetch_soup(url)
        if soup is None:
            continue

        # =========================
        # Selector sesuai versi Anda
        # =========================
        articles = soup.find_all('a', class_='article-link')

        if not articles:
            print(f"Tidak ada artikel ditemukan di page {page}. Stop.")
            break

        for art in articles:
            try:
                title = art.find('h2').text.strip()
                link = art['href']

                # selector tanggal sesuai versi Anda (utama) + fallback defensif
                time_div = art.find('div', class_='articlePost-date')
                if time_div is None:
                    time_div = art.find('div', class_='article__date')
                time_info = time_div.text.strip() if time_div else ''

                content = get_article_kompas(link)

                batch.append([title, time_info, content])
                counter += 1

                # logging ringan
                if counter % 25 == 0:
                    print(f"  - progress batch: {len(batch)} | total di run ini: {counter}")

                # Checkpoint tiap 100
                if len(batch) >= BATCH_SIZE:
                    append_rows_to_csv(CSV_FILENAME, CSV_HEADERS, batch)
                    saved_count += len(batch)
                    batch.clear()

                    if total_results:
                        print(f"💾 tersimpan: {saved_count} dari {total_results}")
                    else:
                        print(f"💾 tersimpan: {saved_count} (total hasil tidak terdeteksi)")

                time.sleep(SLEEP_SEC)

            except Exception as e:
                print(f"Error parsing artikel di halaman {page} | {e}")
                continue

    except Exception as e:
        print(f"Error scraping di halaman {page} | {e}")
        continue

# flush sisa batch
if batch:
    append_rows_to_csv(CSV_FILENAME, CSV_HEADERS, batch)
    saved_count += len(batch)
    batch.clear()

print("\nSelesai.")
if total_results:
    print(f"Total tersimpan: {saved_count} dari {total_results}")
else:
    print(f"Total tersimpan: {saved_count}")



Scraping halaman ke-1: https://search.kompas.com/search?q=politik+indonesia&sort=latest&site_id=1&last_date=all&page=1

Scraping halaman ke-2: https://search.kompas.com/search?q=politik+indonesia&sort=latest&site_id=1&last_date=all&page=2
  - progress batch: 25 | total di run ini: 25

Scraping halaman ke-3: https://search.kompas.com/search?q=politik+indonesia&sort=latest&site_id=1&last_date=all&page=3
  - progress batch: 50 | total di run ini: 50

Scraping halaman ke-4: https://search.kompas.com/search?q=politik+indonesia&sort=latest&site_id=1&last_date=all&page=4
  - progress batch: 75 | total di run ini: 75

Scraping halaman ke-5: https://search.kompas.com/search?q=politik+indonesia&sort=latest&site_id=1&last_date=all&page=5
  - progress batch: 100 | total di run ini: 100
💾 tersimpan: 100 dari 29139

Scraping halaman ke-6: https://search.kompas.com/search?q=politik+indonesia&sort=latest&site_id=1&last_date=all&page=6

Scraping halaman ke-7: https://search.kompas.com/search?q=politik

In [6]:
# Verifikasi cepat hasil CSV
import pandas as pd

if os.path.exists(CSV_FILENAME) and os.path.getsize(CSV_FILENAME) > 0:
    df = pd.read_csv(CSV_FILENAME)
    print("Row di CSV:", len(df))
    display(df.head(3))
    display(df.tail(3))
else:
    print("CSV belum ada / kosong.")


Row di CSV: 9980


,title,tanggal,content
0,Menko Polkam Pastikan Perayaan Malam Tahun Bar...,31 Desember 2025,"JAKARTA, KOMPAS.com -Menteri Koordinator Bidan..."
1,Indonesia 2025: Potret Serdadu dan Polisi Maju...,31 Desember 2025,Di BALIKekonomi yang tidak bisa disebut baik-b...
2,Bagaimana Bisa 68 Anak Indonesia Terpapar Whit...,31 Desember 2025,"JAKARTA, KOMPAS.com- Polisi mengungkap ada pul..."


,title,tanggal,content
9977,"UPDATE 20 Januari: Sebaran 2.116, Kasus Harian...",20 Januari 2022,"JAKARTA, KOMPAS.com -Pemerintah memperbarui in..."
9978,"UPDATE: Bertambah 2.116, Kasus Covid-19 di Ind...",20 Januari 2022,"JAKARTA, KOMPAS.com- Satuan Tugas (Satgas) Pen..."
9979,"Sebut Transformasi Energi Butuh Dana Besar, Jo...",20 Januari 2022,"JAKARTA, KOMPAS.com -Presiden Joko Widodo meng..."
